# Trening c-ResUnet modela na Google Colab GPU

Ovaj notebook je **self-contained** — sav kod (`models.py`, `train.py`, `data_loading.py`, `config.py`) i zamrznuti split fajlovi su ugrađeni direktno u ćelije.

**Pre pokretanja:**
1. Postavi Runtime → Change runtime type → **T4 GPU**
2. Dataset `fluocells/` mora biti na tvom Google Drive-u. Ako imaš samo deljeni link, otvori ga i klikni **Organize → Add shortcut** (ili desni klik → Add shortcut to Drive) da se pojavi u `My Drive`.
3. Podesi `DRIVE_FLUOCELLS_PATH` u ćeliji ispod da pokazuje na tvoj fluocells folder.

**Šta radi:**
- Trenira **M0** (baseline c-ResUnet), **M1** (attention+ELU), **M2** (attention+ReLU)
- Koristi **mixed precision (AMP)** za ~2x ubrzanje na T4
- Čuva modele i predikcije na Google Drive
- Ukupno vreme: ~30-60min na T4 GPU

In [ ]:
#@title ⚙️ Konfiguracija — podesi pre pokretanja

# Putanja do fluocells foldera na tvom Google Drive-u
# (folder koji sadrži all_images/ i all_masks/)
DRIVE_FLUOCELLS_PATH = "/content/drive/MyDrive/fluocells"  #@param {type:"string"}

# Gde čuvamo rezultate (modeli + predikcije) na Drive-u
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/projekat_rezultati"  #@param {type:"string"}

---
## 0. Setup: Drive, paketi, direktorijumi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q albumentations

In [ ]:
import os
from pathlib import Path

# Proveri dataset
fluocells = Path(DRIVE_FLUOCELLS_PATH)
img_dir = fluocells / "all_images" / "images"
mask_dir = fluocells / "all_masks" / "masks"

assert img_dir.exists(), f"Ne postoji: {img_dir}"
assert mask_dir.exists(), f"Ne postoji: {mask_dir}"

n_img = len(list(img_dir.glob("*.png")))
n_mask = len(list(mask_dir.glob("*.png")))
print(f"Dataset OK: {n_img} slika, {n_mask} maski")

# Radni direktorijum
WORK = Path("/content/cell_counting")
WORK.mkdir(exist_ok=True)
(WORK / "src").mkdir(exist_ok=True)
(WORK / "data" / "splits").mkdir(parents=True, exist_ok=True)
(WORK / "models").mkdir(exist_ok=True)
(WORK / "preds").mkdir(exist_ok=True)

# Output na Drive
Path(DRIVE_OUTPUT_PATH).mkdir(parents=True, exist_ok=True)

os.chdir(WORK)
print(f"Radni direktorijum: {WORK}")

---
## 1. Ugrađeni source fajlovi

In [ ]:
%%writefile src/__init__.py
# prazan

In [ ]:
%%writefile src/config.py
from pathlib import Path

ROOT = Path("/content/cell_counting")

# Putanje — dataset sa Drive-a, ostalo lokalno
DATA_DIR = None  # postavljamo spolja
IMG_DIR = None
MASK_DIR = None
SPLITS_DIR = ROOT / "data" / "splits"

SEED = 42

CROP_SIZE = 512
CROP_STARTS_X = [0, 288, 688, 1088]
CROP_STARTS_Y = [0, 288, 688]

MIN_OBJECT_SIZE = 90

# Trening
MODELS_DIR = ROOT / "models"
PREDS_DIR = ROOT / "preds"

LEARNING_RATE = 1e-4
BATCH_SIZE_BASELINE = 16
BATCH_SIZE_ATTENTION = 8
NUM_EPOCHS = 50
EARLY_STOP_PATIENCE = 10
LR_PATIENCE = 4
LR_FACTOR = 0.7

W_POSITIVE = 1.5
W_NEGATIVE = 0.5

DATA_FRACTIONS = [0.25, 0.50, 0.75, 1.0]

In [ ]:
# Postavi putanje do dataseta na Drive-u
import src.config as config
config.DATA_DIR = Path(DRIVE_FLUOCELLS_PATH)
config.IMG_DIR = config.DATA_DIR / "all_images" / "images"
config.MASK_DIR = config.DATA_DIR / "all_masks" / "masks"
print(f"IMG_DIR: {config.IMG_DIR}")
print(f"MASK_DIR: {config.MASK_DIR}")

In [ ]:
%%writefile src/data_loading.py
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

from src import config

CROP_WINDOWS = [(x, y) for x in config.CROP_STARTS_X for y in config.CROP_STARTS_Y]


def _train_aug():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=90, border_mode=0, p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.GaussNoise(p=0.3),
        A.ElasticTransform(p=0.3),
    ])


def _to_tensor():
    return A.Compose([
        A.Normalize(mean=(0, 0, 0), std=(1, 1, 1), max_pixel_value=255.0),
        ToTensorV2(),
    ])


class CellDataset(Dataset):
    def __init__(self, image_ids, form="full", augment=False, cache=True):
        self.form = form
        self.aug = _train_aug() if augment else None
        self.to_tensor = _to_tensor()
        self._use_cache = cache
        self._img_cache, self._mask_cache = {}, {}
        if form == "crops":
            self.samples = [(iid, ci) for iid in image_ids for ci in range(len(CROP_WINDOWS))]
        else:
            self.samples = [(iid, None) for iid in image_ids]

    def _load_full(self, image_id):
        if self._use_cache and image_id in self._img_cache:
            return self._img_cache[image_id], self._mask_cache[image_id]
        img = np.array(Image.open(config.IMG_DIR / image_id).convert("RGB"))
        mask = (np.array(Image.open(config.MASK_DIR / image_id)) > 0).astype(np.uint8)
        if self._use_cache:
            self._img_cache[image_id] = img
            self._mask_cache[image_id] = mask
        return img, mask

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        iid, ci = self.samples[idx]
        img, mask = self._load_full(iid)
        if self.form == "crops":
            x, y = CROP_WINDOWS[ci]
            img = img[y:y + config.CROP_SIZE, x:x + config.CROP_SIZE]
            mask = mask[y:y + config.CROP_SIZE, x:x + config.CROP_SIZE]
        if self.aug is not None:
            out = self.aug(image=img, mask=mask)
            img, mask = out["image"], out["mask"]
        out = self.to_tensor(image=img, mask=mask)
        image_t = out["image"].float()
        mask_t = out["mask"].unsqueeze(0).float()
        return image_t, mask_t, iid


def _read_split(split):
    path = config.SPLITS_DIR / f"{split}_image_ids.txt"
    return [ln.strip() for ln in path.read_text().splitlines() if ln.strip()]


def make_dataset(split=None, image_ids=None, form="full", augment=False, cache=True):
    if image_ids is None:
        image_ids = _read_split(split)
    return CellDataset(image_ids, form=form, augment=augment, cache=cache)

In [ ]:
%%writefile src/models.py
import torch
import torch.nn as nn


def _activation(name):
    if name == "elu":
        return nn.ELU(inplace=True)
    elif name == "relu":
        return nn.ReLU(inplace=True)
    raise ValueError(f"Nepoznata aktivacija: {name}")


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, activation="elu"):
        super().__init__()
        self.conv_branch = nn.Sequential(
            nn.BatchNorm2d(in_ch),
            _activation(activation),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            _activation(activation),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        )
        self.shortcut = (
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
            if in_ch != out_ch
            else nn.Identity()
        )

    def forward(self, x):
        return self.conv_branch(x) + self.shortcut(x)


class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, bias=False),
            nn.BatchNorm2d(F_int),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, bias=False),
            nn.BatchNorm2d(F_int),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class CResUnet(nn.Module):
    def __init__(self, in_channels=3, activation="elu", use_attention=False):
        super().__init__()
        self.use_attention = use_attention
        act = activation

        self.init_conv = nn.Conv2d(in_channels, 16, kernel_size=1, bias=False)

        self.enc1 = ResBlock(16, 16, act)
        self.enc1_5x5 = nn.Sequential(
            nn.BatchNorm2d(16),
            _activation(act),
            nn.Conv2d(16, 16, kernel_size=5, padding=2, bias=False),
        )
        self.pool1 = nn.MaxPool2d(2, stride=2)

        self.enc2 = ResBlock(16, 32, act)
        self.pool2 = nn.MaxPool2d(2, stride=2)

        self.enc3 = ResBlock(32, 64, act)
        self.pool3 = nn.MaxPool2d(2, stride=2)

        self.bottleneck = ResBlock(64, 128, act)

        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2, bias=False)
        self.dec3 = ResBlock(128, 64, act)

        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2, bias=False)
        self.dec2 = ResBlock(64, 32, act)

        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2, bias=False)
        self.dec1 = ResBlock(32, 16, act)

        self.out_conv = nn.Sequential(
            nn.Conv2d(16, 1, kernel_size=1),
            nn.Sigmoid(),
        )

        if use_attention:
            self.ag3 = AttentionGate(F_g=64, F_l=64, F_int=32)
            self.ag2 = AttentionGate(F_g=32, F_l=32, F_int=16)
            self.ag1 = AttentionGate(F_g=16, F_l=16, F_int=8)

    def forward(self, x):
        x0 = self.init_conv(x)

        e1 = self.enc1(x0)
        e1 = e1 + self.enc1_5x5(e1)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        e3 = self.enc3(p2)
        p3 = self.pool3(e3)

        b = self.bottleneck(p3)

        d3 = self.up3(b)
        skip3 = self.ag3(g=d3, x=e3) if self.use_attention else e3
        d3 = torch.cat([d3, skip3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        skip2 = self.ag2(g=d2, x=e2) if self.use_attention else e2
        d2 = torch.cat([d2, skip2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        skip1 = self.ag1(g=d1, x=e1) if self.use_attention else e1
        d1 = torch.cat([d1, skip1], dim=1)
        d1 = self.dec1(d1)

        return self.out_conv(d1)


def build_model(attention=False, activation="elu", in_channels=3):
    model = CResUnet(in_channels=in_channels, activation=activation, use_attention=attention)
    model.apply(init_weights)
    return model


def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)

In [ ]:
%%writefile src/train.py
import copy
import numpy as np
import torch
from pathlib import Path
from tqdm.notebook import tqdm

from src import config


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def weighted_bce_loss(pred, target, w_p=None, w_n=None):
    if w_p is None:
        w_p = config.W_POSITIVE
    if w_n is None:
        w_n = config.W_NEGATIVE
    eps = 1e-7
    loss = -w_p * target * torch.log(pred + eps) \
           -w_n * (1 - target) * torch.log(1 - pred + eps)
    return loss.mean()


def train_model(model, train_loader, val_loader, num_epochs=None, lr=None, device=None):
    if num_epochs is None:
        num_epochs = config.NUM_EPOCHS
    if lr is None:
        lr = config.LEARNING_RATE
    if device is None:
        device = get_device()

    use_amp = (device.type == "cuda")

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=config.LR_PATIENCE, factor=config.LR_FACTOR,
    )
    scaler = torch.amp.GradScaler(enabled=use_amp)

    best_val_loss = float("inf")
    best_state = None
    best_epoch = -1
    epochs_no_improve = 0

    metrics = {"train_loss": [], "val_loss": [], "epoch": [], "lr": []}

    pbar = tqdm(total=num_epochs, desc="Trening")

    for epoch in range(num_epochs):
        model.train()
        running_loss, n_samples = 0.0, 0
        for images, masks, _ in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                pred = model(images)
                loss = weighted_bce_loss(pred, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * images.size(0)
            n_samples += images.size(0)

        train_loss = running_loss / n_samples

        model.eval()
        val_running, val_n = 0.0, 0
        with torch.no_grad():
            for images, masks, _ in val_loader:
                images, masks = images.to(device), masks.to(device)
                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    pred = model(images)
                    loss = weighted_bce_loss(pred, masks)
                val_running += loss.item() * images.size(0)
                val_n += images.size(0)

        val_loss = val_running / val_n
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        metrics["train_loss"].append(train_loss)
        metrics["val_loss"].append(val_loss)
        metrics["epoch"].append(epoch)
        metrics["lr"].append(current_lr)

        pbar.set_postfix({"train": f"{train_loss:.4f}", "val": f"{val_loss:.4f}",
                          "lr": f"{current_lr:.2e}", "best_ep": best_epoch})
        pbar.update(1)

        if epochs_no_improve >= config.EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping na epohi {epoch} "
                  f"(best = ep {best_epoch}, val_loss = {best_val_loss:.4f})")
            break

    pbar.close()
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"Najbolja epoha: {best_epoch}, val_loss: {best_val_loss:.4f}")
    return model, metrics


def _pixel_f1(pred_binary, gt_binary):
    tp = (pred_binary * gt_binary).sum()
    fp = (pred_binary * (1 - gt_binary)).sum()
    fn = ((1 - pred_binary) * gt_binary).sum()
    if tp == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return float(2 * precision * recall / (precision + recall))


def find_best_threshold(model, val_loader, device=None, thresholds=None):
    if device is None:
        device = get_device()
    if thresholds is None:
        thresholds = np.arange(0.05, 0.95, 0.05)

    use_amp = (device.type == "cuda")
    model.eval()
    all_preds, all_gts = [], []
    with torch.no_grad():
        for images, masks, _ in val_loader:
            images = images.to(device)
            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                pred = model(images)
            all_preds.append(pred.float().cpu())
            all_gts.append(masks)

    all_preds = torch.cat(all_preds)
    all_gts = torch.cat(all_gts)

    best_thresh, best_f1 = 0.5, 0.0
    results = []
    for t in thresholds:
        binary = (all_preds > t).float()
        f1 = _pixel_f1(binary, all_gts)
        results.append((t, f1))
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    print(f"Najbolji threshold: {best_thresh:.2f} (piksel F1 = {best_f1:.4f})")
    return best_thresh, results


def save_model(model, model_name, threshold):
    save_dir = config.MODELS_DIR
    save_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), save_dir / f"{model_name}.pth")
    (save_dir / f"{model_name}_threshold.txt").write_text(f"{threshold:.4f}\n")
    print(f"Model sacuvan: {save_dir / model_name}.pth (threshold={threshold:.4f})")


def save_predictions(model, test_loader, model_name, device=None):
    if device is None:
        device = get_device()
    use_amp = (device.type == "cuda")
    pred_dir = config.PREDS_DIR / model_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    model.eval()
    with torch.no_grad():
        for images, _, image_ids in tqdm(test_loader, desc="Predikcije"):
            images = images.to(device)
            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                preds = model(images)
            preds = preds.float().cpu().numpy()
            for i, iid in enumerate(image_ids):
                heatmap = preds[i, 0]
                np.save(pred_dir / f"{iid}.npy", heatmap)
    print(f"Predikcije sacuvane u {pred_dir} ({len(test_loader.dataset)} slika)")


def get_subset_ids(train_ids, fraction, seed=42):
    rng = np.random.RandomState(seed)
    n = max(1, int(len(train_ids) * fraction))
    indices = rng.choice(len(train_ids), size=n, replace=False)
    return [train_ids[i] for i in sorted(indices)]


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    for name, layer in model.named_children():
        n = sum(p.numel() for p in layer.parameters())
        print(f"  {name}: {n:,} parametara")
    print(f"  UKUPNO: {total:,}")
    return total

---
## 2. Ugrađeni split fajlovi

In [ ]:
TRAIN_IDS = """37_y.png
MAR38S1C3R1_DML_20_o.png
MAR38S1C3R1_DMR_20_o.png
MAR39S2C2R2_DML_200x_o.png
MAR39S2C2R2_DMR_200x_o.png
MAR52S2C1R3_LHL_20_o.png
MAR55S3C2R2_VLPAGL_20_o.png
MAR55S3C2R2_VLPAGR_20_o.png
Mar19bS1C1R3_VLPAGr_200x_y.png
Mar19bS1C2R2_VLPAGl_200x_y.png
Mar19bS1C2R2_VLPAGr_200x_y.png
Mar19bS1C2R3_VLPAGl_200x_y.png
Mar19bS1C3R1_VLPAGl_200x_y.png
Mar19bS1C3R3_VLPAGl_200x_y.png
Mar19bS1C3R3_VLPAGr_200x_y.png
Mar19bS1C4R2_DMl_200x_y.png
Mar19bS1C4R2_DMr_200x_y.png
Mar19bS1C4R2_LHl_200x_y.png
Mar19bS1C4R3_DMl_200x_y.png
Mar19bS1C4R3_LHl_200x_y.png
Mar19bS1C5R1_LHl_200x_y.png
Mar19bS1C5R1_LHr_200x_y.png
Mar19bS1C5R2_LHr_200x_y.png
Mar19bS1C5R3_DMl_200x_y.png
Mar19bS1C5R3_LHr_200x_y.png
Mar19bS2C1R1_DMl_200x_y.png
Mar19bS2C1R1_LHl_200x_y.png
Mar19bS2C1R1_LHr_200x_y.png
Mar19bS2C1R2_LHl_200x_y.png
Mar19bS2C1R2_LHr_200x_y.png
Mar19bS2C2R1_LHr_200x_y.png
Mar20bS1C1R3_VLPAGl_200x_y.png
Mar20bS1C2R1_VLPAGr_200x_y.png
Mar20bS1C2R2_VLPAGr_200x_y.png
Mar20bS1C2R3_VLPAGl_200x_y.png
Mar20bS1C3R1_VLPAGl_200x_y.png
Mar20bS1C3R2_VLPAGr_200x_y.png
Mar20bS1C4R1_DMl_200x_y.png
Mar20bS1C4R1_LHr_200x_y.png
Mar20bS1C4R3_LHl_200x_y.png
Mar20bS1C4R3_LHr_200x_y.png
Mar20bS2C1R1_DMl_200x_y.png
Mar20bS2C1R1_DMr_200x_y.png
Mar20bS2C1R1_LHr_200x_y.png
Mar20bS2C1R2_DMl_200x_y.png
Mar20bS2C1R2_DMr_200x_y.png
Mar20bS2C1R2_LHr_200x_y.png
Mar20bS2C1R3_LHl_200x_y.png
Mar20bS2C2R1_LHl_200x_y.png
Mar20bS2C2R2_DMl_200x_y.png
Mar20bS2C2R2_LHr_200x_y.png
Mar20bS2C3R1_LHr_200x_y.png
Mar20bS2C3R2_LHl_200x_y.png
Mar20bS2C3R2_LHr_200x_y.png
Mar21bS1C1R2_VLPAGl_200x_y.png
Mar21bS1C2R2_VLPAGr_200x_y.png
Mar21bS1C2R3_VLPAGl_200x_y.png
Mar21bS1C4R2_DMl_200x_y.png
Mar21bS1C4R2_DMr_200x_y.png
Mar21bS1C4R2_LHr_200x_y.png
Mar21bS2C1R1_LHl_200x_y.png
Mar21bS2C1R2_DMl_200x_y.png
Mar21bS2C1R2_DMr_200x_y.png
Mar21bS2C1R2_LHr_200x_y.png
Mar21bS2C1R3_DMl_200x_y.png
Mar21bS2C1R3_DMr_200x_y.png
Mar21bS2C1R3_LHr_200x_y.png
Mar21bS2C2R1_DMl_200x_y.png
Mar21bS2C2R1_LHl_200x_y.png
Mar21bS2C2R1_LHr_200x_y.png
Mar21bS2C2R2_DMl_200x_y.png
Mar21bS2C2R2_DMr_200x_y.png
Mar21bS2C2R2_LHl_200x_y.png
Mar21bS2C2R3_LHr_200x_y.png
Mar21bS2C3R2_LHl_200x_y.png
Mar21bS2C3R2_LHr_200x_y.png
Mar22bS1C1R3_VLPAGl_200x_y.png
Mar22bS1C1R3_VLPAGr_200x_y.png
Mar22bS1C2R4_DMl_200x_y.png
Mar22bS1C2R4_DMr_200x_y.png
Mar22bS1C3R1_DMl_200x_y.png
Mar22bS1C3R2_DMl_200x_y.png
Mar22bS1C3R3_DMl_200x_y.png
Mar22bS1C3R3_DMr_200x_y.png
Mar22bS1C3R3_LHl_200x_y.png
Mar22bS1C3R3_LHr_200x_y.png
Mar22bS1C4R1_LHr_200x_y.png
Mar22bS1C4R2_DMl_200x_y.png
Mar22bS1C4R2_DMr_200x_y.png
Mar22bS1C4R2_LHl_200x_y.png
Mar22bS1C4R3_DMr_200x_y.png
Mar22bS1C4R3_LHl_200x_y.png
Mar22bS1C4R3_LHr_200x_y.png
Mar22bS1C5R1_LHl_200x_y.png
Mar22bS2C1R1_LHl_200x_y.png
Mar23bS1C2R1_VLPAGr_200x_y.png
Mar23bS1C2R2_VLPAGl_200x_y.png
Mar23bS1C2R2_VLPAGr_200x_y.png
Mar23bS1C2R3_VLPAGl_200x_y.png
Mar23bS1C2R4_VLPAGl_200x_y.png
Mar23bS1C2R4_VLPAGr_200x_y.png
Mar23bS1C5R2_DMl_200x_y.png
Mar23bS1C5R2_DMr_200x_y.png
Mar23bS1C5R2_LHl_200x_y.png
Mar23bS1C5R2_LHr_200x_y.png
Mar23bS1C5R3_DMr_200x_y.png
Mar23bS1C5R3_LHl_200x_y.png
Mar23bS1C6R1_LHl_200x_y.png
Mar23bS1C6R1_LHr_200x_y.png
Mar23bS1C6R2_LHr_200x_y.png
Mar23bS1C6R3_LHl_200x_y.png
Mar23bS1C6R3_LHr_200x_y.png
Mar23bS2C1R1_LHr_200x_y.png
Mar24bS1C1R1_DMl_200x_y.png
Mar24bS1C1R1_DMr_200x_y.png
Mar24bS1C1R1_LHr_200x_y.png
Mar24bS1C1R2_DMl_200x_y.png
Mar24bS1C2R1_LHl_200x_y.png
Mar24bS1C2R1_LHr_200x_y.png
Mar24bS1C2R2_DMl_200x_y.png
Mar24bS1C2R2_DMr_200x_y.png
Mar24bS1C2R2_LHr_200x_y.png
Mar24bS1C2R3_DMr_200x_y.png
Mar24bS1C2R3_LHl_200x_y.png
Mar24bS1C2R3_LHr_200x_y.png
Mar24bS1C3R1_LHl_200x_y.png
Mar24bS1C3R2_LHl_200x_y.png
Mar24bS2C1R3_VLPAGl_200x_y.png
Mar24bS2C1R3_VLPAGr_200x_y.png
Mar24bS2C2R2_VLPAGl_200x_y.png
Mar24bS2C2R2_VLPAGr_200x_y.png
Mar24bS2C4R3_LHl_200x_y.png
Mar24bS2C4R3_LHr_200x_y.png
Mar26bS1C1R3_VLPAGr_200x_y.png
Mar26bS1C1R4_VLPAGr_200x_y.png
Mar26bS1C2R1_VLPAGl_200x_y.png
Mar26bS1C2R1_VLPAGr_200x_y.png
Mar26bS1C2R2_VLPAGl_200x_y.png
Mar26bS1C2R2_VLPAGr_200x_y.png
Mar26bS1C2R3_VLPAGl_200x_y.png
Mar26bS1C2R3_VLPAGr_200x_y.png
Mar26bS1C4R2_DMl_200x_y.png
Mar26bS1C4R2_DMr_200x_y.png
Mar26bS1C4R2_LHr_200x_y.png
Mar26bS1C4R3_DMr_200x_y.png
Mar26bS1C4R3_LHl_200x_y.png
Mar26bS1C4R3_LHr_200x_y.png
Mar26bS2C1R1_LHl_200x_y.png
Mar26bS2C1R1_LHr_200x_y.png
Mar26bS2C1R2_DMr_200x_y.png
Mar26bS2C1R2_LHr_200x_y.png
Mar26bS2C2R1_LHl_200x_y.png
Mar26bS2C2R1_LHr_200x_y.png
Mar26bS2C2R2_DMl_200x_y.png
Mar26bS2C2R2_LHl_200x_y.png
Mar26bS2C2R3_LHl_200x_y.png
Mar26bS2C2R3_LHr_200x_y.png
Mar26bS2C3R1_LHl_200x_y.png
Mar26bS2C3R1_LHr_200x_y.png
Mar27bS1C1R3_VLPAGr_200x_y.png
Mar27bS1C2R2_LHr_200x_y.png
Mar27bS1C2R3_LHr_200x_y.png
Mar27bS1C3R1_LHl_200x_y.png
Mar33bS1C4R2_DMr_200x_y.png
Mar37S1C2R1_DMl_200x_o.png
Mar37S1C2R1_DMr_200x_o.png
Mar40S1C2R2_DMl_200x_o.png
Mar40S3C4R2_VLPAGr_200x_o.png
Mar41S3C1R1_DMl_200x_o.png
Mar41S3C3R3_VLPAGl_200x_o.png"""

VAL_IDS = """38_y.png
Mar19bS1C4R1_VLPAGl_200x_y.png
Mar19bS1C5R1_DMl_200x_y.png
Mar19bS1C5R1_DMr_200x_y.png
Mar19bS2C1R1_DMr_200x_y.png
Mar19bS2C2R1_LHl_200x_y.png
Mar20bS1C3R1_VLPAGr_200x_y.png
Mar20bS2C1R2_LHl_200x_y.png
Mar20bS2C1R3_DMr_200x_y.png
Mar20bS2C1R3_LHr_200x_y.png
Mar20bS2C2R1_LHr_200x_y.png
Mar20bS2C2R2_DMr_200x_y.png
Mar20bS2C2R2_LHl_200x_y.png
Mar20bS2C3R1_LHl_200x_y.png
Mar21bS1C1R2_VLPAGr_200x_y.png
Mar21bS1C1R3_VLPAGl_200x_y.png
Mar21bS1C2R1_VLPAGr_200x_y.png
Mar21bS1C4R2_LHl_200x_y.png
Mar21bS2C1R3_LHl_200x_y.png
Mar21bS2C2R1_DMr_200x_y.png
Mar22bS1C4R3_DMl_200x_y.png
Mar22bS1C5R1_LHr_200x_y.png
Mar23bS1C1R4_VLPAGr_200x_y.png
Mar23bS1C2R1_VLPAGl_200x_y.png
Mar23bS1C2R3_VLPAGr_200x_y.png
Mar23bS1C5R3_DMl_200x_y.png
Mar23bS1C5R3_LHr_200x_y.png
Mar23bS1C6R1_DMl_200x_y.png
Mar23bS1C6R2_DMr_200x_y.png
Mar23bS1C6R2_LHl_200x_y.png
Mar24bS1C1R2_LHl_200x_y.png
Mar24bS1C2R3_DMl_200x_y.png
Mar24bS2C2R3_VLPAGr_200x_y.png
Mar24bS2C4R3_DMl_200x_y.png
Mar26bS1C1R4_VLPAGl_200x_y.png
Mar26bS2C1R1_DMl_200x_y.png
Mar26bS2C1R2_DMl_200x_y.png
Mar26bS2C1R2_LHl_200x_y.png
Mar27bS1C2R1_LHr_200x_y.png
Mar36bS1C6R2_DMl_200x_y.png
Mar40S1C2R2_DMr_200x_o.png
Mar41S3C1R1_DMr_200x_o.png
Mar43S1C5R3_DMr_200x_o.png"""

TEST_IDS = """39_y.png
MAR38S1C3R1_LHR_20_o.png
MAR55S1C5R3_DMR_20_o.png
Mar19bS1C1R2_VLPAGr_200x_y.png
Mar19bS1C1R3_VLPAGl_200x_y.png
Mar19bS1C2R3_VLPAGr_200x_y.png
Mar19bS1C3R2_VLPAGl_200x_y.png
Mar19bS1C3R2_VLPAGr_200x_y.png
Mar19bS1C4R1_VLPAGr_200x_y.png
Mar19bS1C4R2_LHr_200x_y.png
Mar19bS1C4R3_DMr_200x_y.png
Mar19bS1C4R3_LHr_200x_y.png
Mar19bS1C5R2_DMl_200x_y.png
Mar19bS1C5R2_DMr_200x_y.png
Mar19bS1C5R2_LHl_200x_y.png
Mar19bS1C5R3_DMr_200x_y.png
Mar19bS1C5R3_LHl_200x_y.png
Mar20bS1C1R3_VLPAGr_200x_y.png
Mar20bS1C2R1_VLPAGl_200x_y.png
Mar20bS1C2R2_VLPAGl_200x_y.png
Mar20bS1C2R3_VLPAGr_200x_y.png
Mar20bS1C3R2_VLPAGl_200x_y.png
Mar20bS1C4R1_DMr_200x_y.png
Mar20bS1C4R1_LHl_200x_y.png
Mar20bS1C4R3_DMl_200x_y.png
Mar20bS1C4R3_DMr_200x_y.png
Mar20bS2C1R1_LHl_200x_y.png
Mar20bS2C1R3_DMl_200x_y.png
Mar20bS2C2R3_LHl_200x_y.png
Mar20bS2C2R3_LHr_200x_y.png
Mar21bS1C1R3_VLPAGr_200x_y.png
Mar21bS1C2R1_VLPAGl_200x_y.png
Mar21bS1C2R2_VLPAGl_200x_y.png
Mar21bS1C2R3_VLPAGr_200x_y.png
Mar21bS2C1R1_LHr_200x_y.png
Mar21bS2C1R2_LHl_200x_y.png
Mar21bS2C2R2_LHr_200x_y.png
Mar21bS2C2R3_LHl_200x_y.png
Mar22bS1C2R1_VLPAGl_200x_y.png
Mar22bS1C3R2_DMr_200x_y.png
Mar22bS1C4R1_LHl_200x_y.png
Mar22bS1C4R2_LHr_200x_y.png
Mar22bS2C1R1_LHr_200x_y.png
Mar23bS1C1R4_VLPAGl_200x_y.png
Mar23bS1C6R1_DMr_200x_y.png
Mar23bS2C1R1_LHl_200x_y.png
Mar24bS1C1R1_LHl_200x_y.png
Mar24bS1C1R2_DMr_200x_y.png
Mar24bS1C2R1_DMl_200x_y.png
Mar24bS1C2R2_LHl_200x_y.png
Mar24bS1C3R1_LHr_200x_y.png
Mar24bS1C3R2_LHr_200x_y.png
Mar24bS2C2R3_VLPAGl_200x_y.png
Mar24bS2C4R3_DMr_200x_y.png
Mar26bS1C4R2_LHl_200x_y.png
Mar26bS1C4R3_DMl_200x_y.png
Mar26bS2C1R1_DMr_200x_y.png
Mar26bS2C2R1_DMl_200x_y.png
Mar26bS2C2R2_DMr_200x_y.png
Mar26bS2C2R2_LHr_200x_y.png
Mar27bS1C2R1_LHl_200x_y.png
Mar27bS1C3R1_LHr_200x_y.png
Mar31bS2C1R2_VLPAGr_200x_y.png
Mar31bS2C3R4_DMr_200x_y.png
Mar32bS2C2R2_DMl_200x_y.png
Mar33bS1C4R2_DMl_200x_y.png
Mar33bS2C1R1_DMl_200x_y.png
Mar36bS1C6R2_DMr_200x_y.png
Mar42S2C2R2_DMr_200x_o.png
Mar42S2C4R2_VLPAGr_200x_o.png"""

# Sačuvaj split fajlove
from pathlib import Path
splits_dir = Path("/content/cell_counting/data/splits")
(splits_dir / "train_image_ids.txt").write_text(TRAIN_IDS.strip() + "\n")
(splits_dir / "val_image_ids.txt").write_text(VAL_IDS.strip() + "\n")
(splits_dir / "test_image_ids.txt").write_text(TEST_IDS.strip() + "\n")

print(f"Train: {len(TRAIN_IDS.strip().splitlines())} slika")
print(f"Val:   {len(VAL_IDS.strip().splitlines())} slika")
print(f"Test:  {len(TEST_IDS.strip().splitlines())} slika")

---
## 3. Reload modula i provera

In [ ]:
import importlib
import src.config, src.data_loading, src.models, src.train
importlib.reload(src.config)

# Ponovo postavi putanje posle reload-a
src.config.DATA_DIR = Path(DRIVE_FLUOCELLS_PATH)
src.config.IMG_DIR = src.config.DATA_DIR / "all_images" / "images"
src.config.MASK_DIR = src.config.DATA_DIR / "all_masks" / "masks"

importlib.reload(src.data_loading)
importlib.reload(src.models)
importlib.reload(src.train)

from src.data_loading import make_dataset, _read_split
from src.models import build_model
from src.train import (
    get_device, train_model, find_best_threshold,
    save_model, save_predictions, count_parameters, get_subset_ids,
)
from src import config

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

device = get_device()
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

In [ ]:
# Smoke test
train_ds = make_dataset(split="train", form="crops", augment=True)
val_ds   = make_dataset(split="val",   form="full",  augment=False)
test_ds  = make_dataset(split="test",  form="full",  augment=False)
print(f"Train: {len(train_ds)} crop-ova | Val: {len(val_ds)} slika | Test: {len(test_ds)} slika")

loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=4)
img, mask, iid = next(iter(loader))
print(f"Batch OK: img={tuple(img.shape)}, mask={tuple(mask.shape)}")

---
## 4. Trening M0 — Baseline c-ResUnet (bez attention)

In [ ]:
_pin = torch.cuda.is_available()

train_loader_m0 = DataLoader(train_ds, batch_size=config.BATCH_SIZE_BASELINE,
                              shuffle=True, num_workers=4, pin_memory=_pin)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=4, pin_memory=_pin)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=4, pin_memory=_pin)

model_m0 = build_model(attention=False, activation="elu").to(device)
print("=== M0: Baseline c-ResUnet (ELU) ===")
print(f"AMP: {device.type == 'cuda'} | Batch: {config.BATCH_SIZE_BASELINE} | Workers: 4")
count_parameters(model_m0)

In [ ]:
model_m0, metrics_m0 = train_model(model_m0, train_loader_m0, val_loader, device=device)

In [ ]:
best_thresh_m0, thresh_results_m0 = find_best_threshold(model_m0, val_loader, device=device)
save_model(model_m0, "M0_baseline", best_thresh_m0)
save_predictions(model_m0, test_loader, "M0_baseline", device=device)

---
## 5. Trening M1 — Attention + ELU

In [ ]:
train_ds_m1 = make_dataset(split="train", form="crops", augment=True)
train_loader_m1 = DataLoader(train_ds_m1, batch_size=config.BATCH_SIZE_ATTENTION,
                              shuffle=True, num_workers=4, pin_memory=_pin)

model_m1 = build_model(attention=True, activation="elu").to(device)
print("=== M1: Attention + ELU ===")
print(f"AMP: {device.type == 'cuda'} | Batch: {config.BATCH_SIZE_ATTENTION} | Workers: 4")
count_parameters(model_m1)

In [ ]:
model_m1, metrics_m1 = train_model(model_m1, train_loader_m1, val_loader, device=device)

In [ ]:
best_thresh_m1, thresh_results_m1 = find_best_threshold(model_m1, val_loader, device=device)
save_model(model_m1, "M1_attention_elu", best_thresh_m1)
save_predictions(model_m1, test_loader, "M1_attention_elu", device=device)

---
## 6. Trening M2 — Attention + ReLU

In [ ]:
train_ds_m2 = make_dataset(split="train", form="crops", augment=True)
train_loader_m2 = DataLoader(train_ds_m2, batch_size=config.BATCH_SIZE_ATTENTION,
                              shuffle=True, num_workers=4, pin_memory=_pin)

model_m2 = build_model(attention=True, activation="relu").to(device)
print("=== M2: Attention + ReLU ===")
print(f"AMP: {device.type == 'cuda'} | Batch: {config.BATCH_SIZE_ATTENTION} | Workers: 4")
count_parameters(model_m2)

In [ ]:
model_m2, metrics_m2 = train_model(model_m2, train_loader_m2, val_loader, device=device)

In [ ]:
best_thresh_m2, thresh_results_m2 = find_best_threshold(model_m2, val_loader, device=device)
save_model(model_m2, "M2_attention_relu", best_thresh_m2)
save_predictions(model_m2, test_loader, "M2_attention_relu", device=device)

---
## 7. Vizualizacija i poređenje

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for metrics, label in [(metrics_m0, "M0 baseline"), (metrics_m1, "M1 attn+ELU"), (metrics_m2, "M2 attn+ReLU")]:
    axes[0].plot(metrics["epoch"], metrics["train_loss"], label=label)
    axes[1].plot(metrics["epoch"], metrics["val_loss"], label=label)

axes[0].set_title("Train Loss"); axes[0].set_xlabel("Epoha"); axes[0].legend()
axes[1].set_title("Val Loss"); axes[1].set_xlabel("Epoha"); axes[1].legend()

for results, label, thresh in [(thresh_results_m0, "M0", best_thresh_m0),
                                (thresh_results_m1, "M1", best_thresh_m1),
                                (thresh_results_m2, "M2", best_thresh_m2)]:
    ts, f1s = zip(*results)
    axes[2].plot(ts, f1s, marker='o', label=f"{label} (best={thresh:.2f})")

axes[2].set_title("Threshold vs F1 (val)"); axes[2].set_xlabel("Threshold"); axes[2].legend()

plt.tight_layout()
plt.savefig(str(config.MODELS_DIR / "loss_comparison.png"), dpi=150)
plt.show()
print(f"\nNajbolji threshold-ovi: M0={best_thresh_m0:.2f}, M1={best_thresh_m1:.2f}, M2={best_thresh_m2:.2f}")

In [ ]:
# Kvalitativni pregled — iste test slike za sva 3 modela
model_m0.eval(); model_m1.eval(); model_m2.eval()

fig, axes = plt.subplots(3, 5, figsize=(28, 16))
col_titles = ["Slika", "GT maska", "M0 baseline", "M1 attn+ELU", "M2 attn+ReLU"]

sample_iter = iter(test_loader)
for row in range(3):
    img, mask, iid = next(sample_iter)
    with torch.no_grad():
        p0 = model_m0(img.to(device)).cpu()
        p1 = model_m1(img.to(device)).cpu()
        p2 = model_m2(img.to(device)).cpu()

    axes[row, 0].imshow(img[0].permute(1, 2, 0).numpy())
    axes[row, 1].imshow(mask[0, 0].numpy(), cmap="gray")
    axes[row, 2].imshow(p0[0, 0].numpy(), cmap="hot", vmin=0, vmax=1)
    axes[row, 3].imshow(p1[0, 0].numpy(), cmap="hot", vmin=0, vmax=1)
    axes[row, 4].imshow(p2[0, 0].numpy(), cmap="hot", vmin=0, vmax=1)
    axes[row, 0].set_ylabel(iid[0][:25], fontsize=9)

for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=12)
for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Kvalitativni pregled — sva 3 modela", fontsize=14)
plt.tight_layout()
plt.savefig(str(config.MODELS_DIR / "qualitative_comparison.png"), dpi=150)
plt.show()

---
## 8. Kopiranje rezultata na Google Drive

In [ ]:
import shutil

output = Path(DRIVE_OUTPUT_PATH)

# Kopiraj modele
dst_models = output / "models"
if dst_models.exists():
    shutil.rmtree(dst_models)
shutil.copytree(config.MODELS_DIR, dst_models)
print(f"Modeli kopirani u {dst_models}")

# Kopiraj predikcije
dst_preds = output / "preds"
if dst_preds.exists():
    shutil.rmtree(dst_preds)
shutil.copytree(config.PREDS_DIR, dst_preds)
print(f"Predikcije kopirane u {dst_preds}")

# Sačuvaj metrike kao numpy
for name, m in [("M0", metrics_m0), ("M1", metrics_m1), ("M2", metrics_m2)]:
    np.savez(output / f"metrics_{name}.npz", **{k: np.array(v) for k, v in m.items()})

print(f"\nSve sačuvano na Drive: {output}")
print("Sadržaj:")
for f in sorted(output.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to(output)}  ({size_mb:.1f} MB)")

---
## 9. Gotovo!

Rezultati su na tvom Google Drive-u u folderu `projekat_rezultati/`.

**Šta da uradiš posle:**
1. Skini folder `projekat_rezultati/` sa Drive-a na svoj računar
2. Kopiraj `models/` i `preds/` u lokalni repo `Fluorescent-neuronal-cell-counting/`
3. Nastavi sa Delom C (post-processing + evaluacija) lokalno — to ne zahteva GPU